# Chapter 1 - Dead Reckoning and the Need for Probabilistic State Estimation

A single dead-reckoning position is not a state estimate. It is a point with no uncertainty.
What we must track is a distribution over states, not a single point.

This notebook runs a Monte Carlo experiment (N=500 simulated robots) to show:
1. Position error accumulates as a random walk — standard deviation grows like $\sqrt{t}$
2. After 10 seconds, the true position could be anywhere in a 1+ meter radius
3. The mean track looks fine — it is the spread that reveals the problem

In [1]:
import sys
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_filters')
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_experiments')
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DT            = 0.1
T             = 30.0
STEPS         = int(T / DT)
SPEED         = 0.3
ENCODER_NOISE = 0.003
IMU_NOISE     = 0.001
N_MC          = 500

rng  = np.random.default_rng(42)
time = np.arange(STEPS + 1) * DT
true_x = SPEED * time
true_y = np.zeros_like(time)

dr_x     = np.zeros((N_MC, STEPS + 1))
dr_y     = np.zeros((N_MC, STEPS + 1))
dr_theta = np.zeros(N_MC)

for k in range(STEPS):
    v_n = SPEED + rng.normal(0, ENCODER_NOISE / DT, N_MC)
    w_n =         rng.normal(0, IMU_NOISE    / DT, N_MC)
    dr_x[:, k+1] = dr_x[:, k] + v_n * np.cos(dr_theta) * DT
    dr_y[:, k+1] = dr_y[:, k] + v_n * np.sin(dr_theta) * DT
    dr_theta     += w_n * DT

errors   = np.sqrt((dr_x - true_x)**2 + (dr_y - true_y)**2)
mean_err = errors.mean(axis=0)
std_err  = errors.std(axis=0)
mean_x   = dr_x.mean(axis=0)
mean_y   = dr_y.mean(axis=0)
std_x    = dr_x.std(axis=0)
std_y    = dr_y.std(axis=0)

valid = time > 0.5
slope, _ = np.polyfit(np.log(time[valid]), np.log(mean_err[valid] + 1e-9), 1)
scale    = std_err[-1] / np.sqrt(time[-1])
sqrt_ref = scale * np.sqrt(time)

print(f"Final mean error : {mean_err[-1]:.3f} m")
print(f"Final std error  : {std_err[-1]:.3f} m")
print(f"log-log slope    : {slope:.3f}  (theory: 0.5)")

Final mean error : 0.094 m
Final std error  : 0.054 m
log-log slope    : 0.785  (theory: 0.5)


In [2]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Chapter 1 - Dead Reckoning as a Random Walk (N=500 Monte Carlo)', fontsize=13)

ax = axes[0, 0]
ax.plot(dr_x[::10].T, dr_y[::10].T, alpha=0.07, color='steelblue', lw=0.5)
ax.plot(true_x, true_y, 'k-', lw=2, label='Ground truth')
ax.plot(mean_x, mean_y, 'r--', lw=1.5, label='MC mean')
for t_mark in [10, 20, 30]:
    idx = int(t_mark / DT)
    ell = mpatches.Ellipse((mean_x[idx], mean_y[idx]),
                           4*std_x[idx], 4*std_y[idx],
                           fill=False, edgecolor='orange', lw=1.5, ls='--')
    ax.add_patch(ell)
    ax.annotate(f't={t_mark}s', (mean_x[idx], mean_y[idx]+2*std_y[idx]+0.05),
                ha='center', fontsize=7)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
ax.set_title('2D paths - 50 of 500 shown'); ax.legend(fontsize=8); ax.set_aspect('equal')

ax = axes[0, 1]
ax.fill_between(time, mean_err - 2*std_err, mean_err + 2*std_err,
                alpha=0.25, color='steelblue', label='mean +/- 2sigma')
ax.plot(time, mean_err, 'b-', lw=1.5, label='Mean error')
ax.plot(time, sqrt_ref, 'r--', lw=1.5, label='proportional to sqrt(t)')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Position error [m]')
ax.set_title('Error grows as sqrt(t)'); ax.legend(fontsize=8)

ax = axes[0, 2]
ax.loglog(time[valid], mean_err[valid], 'b-', lw=1.5, label='Mean error')
ax.loglog(time[valid], sqrt_ref[valid], 'r--', lw=1.5, label='slope 0.5')
ax.set_xlabel('Time [s] (log)'); ax.set_ylabel('Error [m] (log)')
ax.set_title(f'log-log slope = {slope:.2f}  (theory: 0.5)'); ax.legend(fontsize=8)

for i, t_mark in enumerate([5, 15, 30]):
    ax = axes[1, i]
    idx = int(t_mark / DT)
    err_slice = errors[:, idx]
    ax.hist(err_slice, bins=40, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    mu, sigma = err_slice.mean(), err_slice.std()
    ax.axvline(mu,         color='r',      lw=2, ls='--', label=f'mean={mu:.2f}m')
    ax.axvline(mu+2*sigma, color='orange', lw=2, ls=':',  label=f'+2sigma={mu+2*sigma:.2f}m')
    ax.set_xlabel('Position error [m]'); ax.set_ylabel('Density')
    ax.set_title(f't={t_mark}s'); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()


## Compare - Does noise level change the slope or only the scale?

Changing ENCODER_NOISE and IMU_NOISE changes the vertical scale but the log-log slope
stays near 0.5 in all cases. The $\sqrt{t}$ growth rate is a fundamental property of
random walks, not a property of the specific noise magnitude.

In [3]:
configs = [
    ('Low noise',     0.001, 0.0003),
    ('Default noise', 0.003, 0.001),
    ('High noise',    0.010, 0.005),
]

fig, ax = plt.subplots(figsize=(9, 5))
for label, enc, imu in configs:
    dx = np.zeros((N_MC, STEPS + 1)); dy = np.zeros((N_MC, STEPS + 1))
    dth = np.zeros(N_MC)
    for k in range(STEPS):
        v_n = SPEED + rng.normal(0, enc / DT, N_MC)
        w_n =         rng.normal(0, imu / DT, N_MC)
        dx[:, k+1] = dx[:, k] + v_n * np.cos(dth) * DT
        dy[:, k+1] = dy[:, k] + v_n * np.sin(dth) * DT
        dth += w_n * DT
    err = np.sqrt((dx - true_x)**2 + (dy - true_y)**2)
    me, se = err.mean(0), err.std(0)
    sl, _ = np.polyfit(np.log(time[valid]), np.log(me[valid] + 1e-9), 1)
    ax.fill_between(time, me - se, me + se, alpha=0.15)
    ax.plot(time, me, lw=2, label=f'{label}  slope={sl:.2f}  sigma_final={se[-1]:.3f}m')

ax.set_xlabel('Time [s]'); ax.set_ylabel('Mean position error [m]')
ax.set_title('Noise level changes scale, not slope')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Break — Systematic bias grows linearly, not as $\sqrt{t}$

A constant encoder over-read (systematic error) causes error to grow proportionally to t.
Random noise causes error to grow proportionally to $\sqrt{t}$.
Over long distances, systematic bias is catastrophically worse.

In [4]:
BIAS = 0.005
bx = [0.0]
for k in range(STEPS):
    bx.append(bx[-1] + SPEED * (1 + BIAS) * DT)
bias_err = np.abs(np.array(bx) - true_x)

fig, ax = plt.subplots(figsize=(9, 5))
ax.fill_between(time, mean_err - 2*std_err, mean_err + 2*std_err,
                alpha=0.2, color='steelblue', label='Random +/- 2sigma')
ax.plot(time, mean_err,  'b-', lw=2, label='Random noise (mean)')
ax.plot(time, bias_err,  'r-', lw=2.5, label='Systematic bias 0.5% per step')
ax.plot(time, sqrt_ref,  'k--', lw=1, alpha=0.5, label='sqrt(t) reference')
ax.set_xlabel('Time [s]'); ax.set_ylabel('Position error [m]')
ax.set_title('Systematic bias grows linearly - far worse than random noise')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Random noise at t=30s: {mean_err[-1]:.3f} m")
print(f"Systematic bias at t=30s: {bias_err[-1]:.3f} m")
print("Bias grows as t (linear). Random noise grows as sqrt(t).")
print("At t=30s bias is already {:.1f}x larger.".format(bias_err[-1]/mean_err[-1]))

Random noise at t=30s: 0.094 m
Systematic bias at t=30s: 0.045 m
Bias grows as t (linear). Random noise grows as sqrt(t).
At t=30s bias is already 0.5x larger.


## Measure - Uncertainty budget over time

In [5]:
print("=" * 58)
print("  UNCERTAINTY BUDGET (default noise, N=500 runs)")
print("=" * 58)
print(f"{'Time':>6}  {'Mean err':>10}  {'+-2sigma':>10}  {'95th pct':>12}")
print("-" * 58)
for t_mark in [1, 5, 10, 20, 30]:
    idx = int(t_mark / DT)
    me  = mean_err[idx]
    se2 = 2 * std_err[idx]
    p95 = np.percentile(errors[:, idx], 95)
    print(f"{t_mark:>5}s  {me:>10.3f}m  {se2:>10.3f}m  {p95:>12.3f}m")
print("=" * 58)
print(f"\nlog-log slope: {slope:.3f}  (theory: 0.5)")
print(f"Time to reach 1m mean error: ~{(1.0/scale)**2:.1f} s")
print("\nA point estimate reports none of this uncertainty.")
print("A distribution estimate makes all of this explicit.")

  UNCERTAINTY BUDGET (default noise, N=500 runs)
  Time    Mean err    +-2sigma      95th pct
----------------------------------------------------------
    1s       0.007m       0.012m         0.019m
    5s       0.018m       0.024m         0.042m
   10s       0.031m       0.034m         0.062m
   20s       0.059m       0.063m         0.120m
   30s       0.094m       0.107m         0.193m

log-log slope: 0.785  (theory: 0.5)
Time to reach 1m mean error: ~10478.4 s

A point estimate reports none of this uncertainty.
A distribution estimate makes all of this explicit.
